In [19]:
import os
import sys
if os.getcwd().endswith("notebook"):
    os.chdir("..")
sys.path.append("src")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils_cv import baca, kontras_bercak
from praproses import praproses, simpan_param

# ==========================================
# KODE 5.2: Evaluasi Rangkaian Praproses (Tabel 12)
# ==========================================
param = {
    "gamma": 1.0,
    "clahe_clip": 2.0,
    "penapis": "median",
    "kernel": 3
}

# Simpan parameter ke file JSON
simpan_param(param, "hasil/param_praproses.json")
KANAL = 2  # Kanal R terbaik berdasarkan Bagian 2 (indeks BGR: 0=B, 1=G, 2=R)[cite: 32]

label = pd.read_csv("data/sintetis/label.csv")
baris = []

for _, d in label[label.kelas != "sehat"].iterrows():
    img = baca(f"data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"data/sintetis/mask_daun/{d.nama_file}", 0)
    mb = cv2.imread(f"data/sintetis/mask_bercak/{d.nama_file}", 0)

    # Jalankan rangkaian praproses
    out = praproses(img, param)

    baris.append([
        d.kondisi_cahaya,
        kontras_bercak(img[:, :, KANAL], md, mb),
        kontras_bercak(out[:, :, KANAL], md, mb)
    ])

evaluasi = pd.DataFrame(baris, columns=["kondisi", "KB_sebelum", "KB_sesudah"])
tabel_eval = evaluasi.groupby("kondisi").mean().round(2)

print("--- TABEL 12: EVALUASI RANGKAIAN PRAPROSES ---")
print(tabel_eval)

print("\nRangkaian praproses SIMUTEH berhasil dijalankan dan parameter tersimpan di hasil/param_praproses.json!")

--- TABEL 12: EVALUASI RANGKAIAN PRAPROSES ---
                KB_sebelum  KB_sesudah
kondisi                               
bayangan              6.36       10.62
gelap                 9.59       15.96
kontras_rendah        3.55        4.79
normal                9.00       11.44
terang                6.49        6.08

Rangkaian praproses SIMUTEH berhasil dijalankan dan parameter tersimpan di hasil/param_praproses.json!
